In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv("netflix_clean.csv",encoding="latin-1")

Analyze rating categories

In [3]:
rating_counts=df["rating"].value_counts()
print(rating_counts)

rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64


In [4]:
age_map = {
    "TV-Y": "All Kids", "TV-Y7": "Older Kids", "TV-Y7-FV": "Older Kids",
    "TV-G": "General", "G": "General",
    "PG": "Parental Guidance", "TV-PG": "Parental Guidance",
    "PG-13": "Teens (13+)", "TV-14": "Teens (14+)",
    "R": "Mature (17+)", "TV-MA": "Mature (17+)", "NC-17": "Mature (17+)",
    "NR": "Unrated", "UR": "Unrated",
}
df["age_group"] = df["rating"].map(age_map).fillna("Unknown")




In [5]:
age_map_counts = df["age_group"].value_counts()
print(age_map_counts)

age_group
Mature (17+)         4007
Teens (14+)          2157
Parental Guidance    1148
Teens (13+)           490
Older Kids            339
All Kids              306
General               261
Unrated                82
Name: count, dtype: int64


Most common genres (listed_in has several genres per title -> explode)

In [6]:
genres = df[["show_id", "type", "rating","age_group","listed_in"]].copy()
genres["genre"] = genres["listed_in"].str.split(", ")
genres = genres.explode("genre")
genres["genre"] = genres["genre"].str.strip()
top_genres = genres["genre"].value_counts().head(10)
print(top_genres)

genre
International Movies        2752
Dramas                      2426
Comedies                    1674
International TV Shows      1349
Documentaries                869
Action & Adventure           859
TV Dramas                    762
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Name: count, dtype: int64


Compare ratings across content types

In [7]:
rating_type = pd.crosstab(df["rating"], df["type"])
rating_type = rating_type.loc[rating_counts.index]  # keep sorted by popularity
print(rating_type)



type      Movie  TV Show
rating                  
TV-MA      2062     1143
TV-14      1427      730
TV-PG       540      321
R           797        2
PG-13       490        0
TV-Y7       139      194
TV-Y        131      175
PG          287        0
TV-G        126       94
NR           75        4
G            41        0
TV-Y7-FV      5        1
NC-17         3        0
UR            3        0


In [8]:
genre_type = (genres[genres["genre"].isin(top_genres.index)]
.groupby(["genre", "type"]).size().reset_index(name="count"))

In [9]:
import os
import plotly.express as px
os.makedirs("charts", exist_ok=True)
fig1 = px.bar(rating_counts.reset_index(), x="rating", y="count",
              title="Content by Rating", color_discrete_sequence=["red"])
fig1.write_html("charts/task5_ratings.html")

fig2 = px.bar(rating_type.reset_index().melt(id_vars="rating", var_name="type", value_name="count"),
              x="rating", y="count", color="type", barmode="group",
              title="Ratings: Movies vs TV Shows", color_discrete_sequence=["red", "black"])
fig2.write_html("charts/task5_rating_by_type.html")

fig3 = px.bar(top_genres.sort_values().reset_index(), x="count", y="genre", orientation="h",
              title="Top 10 Genres", color_discrete_sequence=["red"])
fig3.write_html("charts/task5_top_genres.html")

fig4 = px.bar(genre_type, x="count", y="genre", color="type", orientation="h", barmode="stack",
              title="Top Genres: Movies vs TV Shows", color_discrete_sequence=["red", "black"])
fig4.write_html("charts/task5_genre_by_type.html")

fig5 = px.pie(age_map_counts.reset_index(), names="age_group", values="count",
              title="Audience Segmentation", hole=0.4)
fig5.write_html("charts/task5_audience.html")

heat = (genres[genres["genre"].isin(top_genres.index)]
        .groupby(["genre", "age_group"]).size().unstack(fill_value=0))
fig6 = px.imshow(heat, text_auto=True, aspect="auto", color_continuous_scale="Reds",
                 title="Genre vs Target Audience")
fig6.write_html("charts/task5_genre_audience_heatmap.html")


In [11]:
for f in [fig1, fig2, fig3, fig4, fig5, fig6]:
    f.show()